# LegalIR Single-T4 Contract Smoke Gate (Google Colab)
**Pinned Runtime Commit:** `33a0930e7d1ca1ee58000efdc28a79cf7107b9bf`

This notebook executes the official single-T4 sequential contract smoke on real canonical v2 data using production modules.
Prerequisites: Commit must have a **GREEN** `LegalIR CI` GitHub Actions run before execution.


In [ ]:
# [1] Hardware Preflight: Verify GPU availability and Tesla T4 contract
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'CUDA is not available! Change runtime type to GPU.'
gpu_name = torch.cuda.get_device_name(0)
print(f'CUDA Device: {gpu_name}')
if 'T4' not in gpu_name:
    print(f'[!] WARNING: Active GPU {gpu_name} is not a Tesla T4. Smoke will run with non-T4 override.')


In [ ]:
# [2] Mount Google Drive and retrieve Colab Secrets
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

# Retrieve HF_TOKEN securely from Colab Secrets (Never print token values)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        os.environ['HF_TOKEN'] = HF_TOKEN
        print('[+] HF_TOKEN loaded securely from Colab Secrets.')
    else:
        print('[!] Warning: HF_TOKEN not found in Secrets. Some private assets may not download.')
except Exception as exc:
    print(f'[!] userdata notice: {exc}')

# Optional GITHUB_TOKEN for high-rate-limit GitHub CI status checking
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    if GITHUB_TOKEN:
        os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
        print('[+] GITHUB_TOKEN loaded securely from Colab Secrets.')
except Exception:
    pass


In [ ]:
# [3] User Configuration: Target Commit SHA and Official Data Path
TARGET_SHA = '33a0930e7d1ca1ee58000efdc28a79cf7107b9bf'
DATA_DIR = '/content/drive/MyDrive/legalir-task1-clean-data'
OUTPUT_DIR = '/content/drive/MyDrive/legalir-smoke-runs/colab_t4_smoke'

print(f'Target SHA : {TARGET_SHA}')
print(f'Data Dir   : {DATA_DIR}')
print(f'Output Dir : {OUTPUT_DIR}')


In [ ]:
# [4] Clone repository and checkout exact TARGET_SHA in detached HEAD
!rm -rf /content/LegalIR
!git clone https://github.com/silent9669/LegalIR.git /content/LegalIR
%cd /content/LegalIR
!git checkout {TARGET_SHA}


In [ ]:
# [5] Install minimal Python dependencies without reinstalling torch
import subprocess, sys
required_pkgs = []
for mod, pkg in [('lightgbm', 'lightgbm'), ('sentencepiece', 'sentencepiece'), ('bm25s', 'bm25s'), ('pyvi', 'pyvi'), ('peft', 'peft'), ('accelerate', 'accelerate'), ('faiss', 'faiss-cpu'), ('psutil', 'psutil')]:
    try:
        __import__(mod)
    except ImportError:
        required_pkgs.append(pkg)
if required_pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-script-location'] + required_pkgs, check=True)


In [ ]:
# [6] Verify GitHub CI Status Gate for TARGET_SHA
!python scripts/verify_github_ci.py --repo silent9669/LegalIR --sha {TARGET_SHA}


In [ ]:
# [7] Execute Official Single-T4 Contract Smoke Gate with all required CLI args
!python scripts/run_colab_t4_smoke.py \
    --data-dir '{DATA_DIR}' \
    --work-dir '{OUTPUT_DIR}' \
    --target-sha '{TARGET_SHA}' \
    --allow-non-t4


In [ ]:
# [8] Display and inspect Colab Smoke Report
import json
from pathlib import Path

report_file = Path(OUTPUT_DIR) / 'colab_smoke_report.json'
if report_file.exists():
    report = json.loads(report_file.read_text(encoding='utf-8'))
    print(json.dumps(report, indent=2))
    print('=' * 65)
    print(f'FINAL COLAB SMOKE VERDICT: {report.get("result")}')
    print('=' * 65)
else:
    print(f'[-] Error: Smoke report not found at {report_file}')
